# Modelo de Machine Learning em NumPy
Notebook que agrega todos os módulos: dados, camadas, activações, losses, optimizer, métricas, vectorizadores de texto, regressão logística, rede neuronal e treino de modelos de texto.

## 1. Imports Globais

In [1]:
import numpy as np
import pandas as pd
import re
import copy
import argparse
import os
import sys
from abc import ABCMeta, abstractmethod

## 2. Data (`data.py`)
Classe principal para encapsular datasets e utilitários de leitura de CSV.

In [2]:
class Data:
    def __init__(self, X, y=None, features=None, label=None):
        if X is None:
            raise ValueError("X cannot be None")
        if y is not None and len(X) != len(y):
            raise ValueError("X and y must have the same length")
        if features is not None and len(X[0]) != len(features):
            raise ValueError("Number of features must match the number of columns in X")
        if features is None:
            features = [f"feat_{str(i)}" for i in range(X.shape[1])]
        if y is not None and label is None:
            label = "y"
        self.X = X
        self.y = y
        self.features = features
        self.label = label

    def shape(self):
        return self.X.shape

    def has_label(self):
        return self.y is not None

    def get_classes(self):
        if self.has_label():
            return np.unique(self.y)
        else:
            raise ValueError("Dataset does not have a label")

    def get_mean(self):
        return np.nanmean(self.X, axis=0)

    def get_variance(self):
        return np.nanvar(self.X, axis=0)

    def get_median(self):
        return np.nanmedian(self.X, axis=0)

    def get_min(self):
        return np.nanmin(self.X, axis=0)

    def get_max(self):
        return np.nanmax(self.X, axis=0)

    def summary(self):
        data = {
            "mean": self.get_mean(),
            "median": self.get_median(),
            "min": self.get_min(),
            "max": self.get_max(),
            "var": self.get_variance()
        }
        return pd.DataFrame.from_dict(data, orient="index", columns=self.features)


def read_csv(filename, sep=',', features=False, label=False):
    data = pd.read_csv(filename, sep=sep, header=None)

    if features and label:
        features = data.columns[:-1]
        label = data.columns[-1]
        X = data.iloc[:, :-1].to_numpy()
        y = data.iloc[:, -1].to_numpy()
    elif features and not label:
        features = data.columns
        X = data.to_numpy()
        y = None
    elif not features and label:
        X = data.iloc[:, :-1].to_numpy()
        y = data.iloc[:, -1].to_numpy()
        features = None
        label = data.columns[-1]
    else:
        X = data.to_numpy()
        y = None
        features = None
        label = None

    return Data(X=X, y=y, features=features, label=label)


# --- Teste rápido ---
X = np.array([[1, 2, 3], [4, 5, 6]])
y = np.array([1, 2])
features = np.array(['a', 'b', 'c'])
dataset = Data(X, y, features, 'y')
print("Shape:", dataset.shape())
dataset.summary()

Shape: (2, 3)


,a,b,c
mean,2.50,3.50,4.50
median,2.50,3.50,4.50
min,1.00,2.00,3.00
max,4.00,5.00,6.00
var,2.25,2.25,2.25


## 3. Optimizer (`optimizer.py`)
SGD com momentum.

In [3]:
class Optimizer:
    def __init__(self, learning_rate=0.01, momentum=0.90):
        self.retained_gradient = None
        self.learning_rate = learning_rate
        self.momentum = momentum

    def update(self, w, grad_loss_w):
        if self.retained_gradient is None:
            self.retained_gradient = np.zeros(np.shape(w))
        self.retained_gradient = (
            self.momentum * self.retained_gradient
            + (1 - self.momentum) * grad_loss_w
        )
        return w - self.learning_rate * self.retained_gradient

## 4. Camadas (`layers.py`)
Camada base abstracta, `DenseLayer` e `DropoutLayer`.

In [4]:
class Layer(metaclass=ABCMeta):

    @abstractmethod
    def forward_propagation(self, input, training):
        raise NotImplementedError

    @abstractmethod
    def backward_propagation(self, error):
        raise NotImplementedError

    @abstractmethod
    def output_shape(self):
        raise NotImplementedError

    @abstractmethod
    def parameters(self):
        raise NotImplementedError

    def set_input_shape(self, input_shape):
        self._input_shape = input_shape

    def input_shape(self):
        return self._input_shape

    def layer_name(self):
        return self.__class__.__name__


class DenseLayer(Layer):
    def __init__(self, n_units, input_shape=None, l2_lambda: float = 0.0):
        super().__init__()
        self.n_units = n_units
        self._input_shape = input_shape
        self.l2_lambda = float(l2_lambda)
        self.input = None
        self.output = None
        self.weights = None
        self.biases = None

    def initialize(self, optimizer):
        self.weights = np.random.rand(self.input_shape()[0], self.n_units) - 0.5
        self.biases = np.zeros((1, self.n_units))
        self.w_opt = copy.deepcopy(optimizer)
        self.b_opt = copy.deepcopy(optimizer)
        return self

    def parameters(self):
        return np.prod(self.weights.shape) + np.prod(self.biases.shape)

    def forward_propagation(self, inputs, training):
        self.input = inputs
        self.output = np.dot(self.input, self.weights) + self.biases
        return self.output

    def backward_propagation(self, output_error):
        input_error = np.dot(output_error, self.weights.T)
        weights_error = np.dot(self.input.T, output_error)
        if self.l2_lambda > 0:
            weights_error = weights_error + self.l2_lambda * self.weights
        bias_error = np.sum(output_error, axis=0, keepdims=True)
        self.weights = self.w_opt.update(self.weights, weights_error)
        self.biases = self.b_opt.update(self.biases, bias_error)
        return input_error

    def output_shape(self):
        return (self.n_units,)


class DropoutLayer(Layer):
    def __init__(self, p_drop: float = 0.5, input_shape=None, seed: int = 42):
        super().__init__()
        if not (0.0 <= p_drop < 1.0):
            raise ValueError("p_drop must be in [0, 1)")
        self.p_drop = float(p_drop)
        self._input_shape = input_shape
        self.seed = seed
        self.rng = np.random.default_rng(seed)
        self.mask = None

    def forward_propagation(self, inputs, training):
        if (not training) or self.p_drop == 0.0:
            self.mask = None
            return inputs
        keep_prob = 1.0 - self.p_drop
        self.mask = (self.rng.random(size=inputs.shape) < keep_prob).astype(inputs.dtype)
        return (inputs * self.mask) / keep_prob

    def backward_propagation(self, output_error):
        if self.mask is None or self.p_drop == 0.0:
            return output_error
        keep_prob = 1.0 - self.p_drop
        return (output_error * self.mask) / keep_prob

    def output_shape(self):
        return self._input_shape

    def parameters(self):
        return 0

## 5. Activações (`activation.py`)
Sigmoid e ReLU.

In [5]:
class ActivationLayer(Layer):

    def forward_propagation(self, input, training):
        self.input = input
        self.output = self.activation_function(self.input)
        return self.output

    def backward_propagation(self, output_error):
        return self.derivative(self.input) * output_error

    @abstractmethod
    def activation_function(self, input):
        raise NotImplementedError

    @abstractmethod
    def derivative(self, input):
        raise NotImplementedError

    def output_shape(self):
        return self._input_shape

    def parameters(self):
        return 0


class SigmoidActivation(ActivationLayer):
    def activation_function(self, input):
        return 1 / (1 + np.exp(-input))

    def derivative(self, input):
        s = self.activation_function(input)
        return s * (1 - s)


class ReLUActivation(ActivationLayer):
    def activation_function(self, input):
        return np.maximum(0, input)

    def derivative(self, input):
        return np.where(input > 0, 1, 0)

## 6. Funções de Loss (`losses.py`)
MSE, Binary Cross-Entropy e Softmax Cross-Entropy.

In [6]:
class LossFunction:
    @abstractmethod
    def loss(self, y_true, y_pred):
        raise NotImplementedError

    @abstractmethod
    def derivative(self, y_true, y_pred):
        raise NotImplementedError


class MeanSquaredError(LossFunction):
    def loss(self, y_true, y_pred):
        return np.mean((y_true - y_pred) ** 2)

    def derivative(self, y_true, y_pred):
        return 2 * (y_pred - y_true) / y_true.shape[0]


class BinaryCrossEntropy(LossFunction):
    def loss(self, y_true, y_pred):
        p = np.clip(y_pred, 1e-15, 1 - 1e-15)
        return -np.mean(y_true * np.log(p) + (1 - y_true) * np.log(1 - p))

    def derivative(self, y_true, y_pred):
        p = np.clip(y_pred, 1e-15, 1 - 1e-15)
        return (p - y_true) / (p * (1 - p) * y_true.shape[0])


class SoftmaxCrossEntropy(LossFunction):
    """Cross-entropy a partir de logits (sem camada softmax separada)."""

    def _one_hot(self, y_true, n_classes):
        y = np.asarray(y_true).astype(int).ravel()
        oh = np.zeros((len(y), n_classes), dtype=np.float32)
        oh[np.arange(len(y)), y] = 1.0
        return oh

    def _log_softmax(self, logits):
        z = logits - np.max(logits, axis=1, keepdims=True)
        logsumexp = np.log(np.sum(np.exp(z), axis=1, keepdims=True))
        return z - logsumexp

    def loss(self, y_true, y_pred):
        logits = np.asarray(y_pred)
        n = logits.shape[0]
        log_probs = self._log_softmax(logits)
        y_oh = self._one_hot(y_true, logits.shape[1]) if np.ndim(y_true) == 1 else np.asarray(y_true)
        return float(-np.sum(y_oh * log_probs) / n)

    def derivative(self, y_true, y_pred):
        logits = np.asarray(y_pred)
        n = logits.shape[0]
        probs = np.exp(self._log_softmax(logits))
        y_oh = self._one_hot(y_true, logits.shape[1]) if np.ndim(y_true) == 1 else np.asarray(y_true)
        return (probs - y_oh) / n

## 7. Métricas (`metrics.py`)

In [7]:
def accuracy(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    if y_true.ndim == 2 and y_true.shape[1] > 1:
        y_true_idx = np.argmax(y_true, axis=1)
    else:
        y_true_idx = y_true.reshape(-1)
        if y_true_idx.ndim != 1:
            y_true_idx = y_true_idx.ravel()

    if y_pred.ndim == 2 and y_pred.shape[1] > 1:
        y_pred_idx = np.argmax(y_pred, axis=1)
    else:
        y_pred_idx = np.round(y_pred.reshape(-1)).astype(int)

    return float(np.mean(y_pred_idx == y_true_idx))


def softmax_accuracy(y_true, logits):
    """Accuracy para multiclasse quando o modelo devolve logits."""
    return accuracy(y_true, logits)


def mse(y_true, y_pred):
    return np.sum((y_true - y_pred) ** 2) / len(y_true)


def mse_derivative(y_true, y_pred):
    return 2 * np.sum(y_true - y_pred) / len(y_true)

## 8. Rede Neuronal (`neuralnet.py`)

In [8]:
class NeuralNetwork:

    def __init__(self, epochs=100, batch_size=128, optimizer=None,
                 learning_rate=0.01, momentum=0.90, verbose=False,
                 loss=MeanSquaredError, metric=mse):
        self.epochs = epochs
        self.batch_size = batch_size
        self.optimizer = Optimizer(learning_rate=learning_rate, momentum=momentum)
        self.verbose = verbose
        self.loss = loss()
        self.metric = metric
        self.layers = []
        self.history = {}

    def add(self, layer):
        if self.layers:
            layer.set_input_shape(input_shape=self.layers[-1].output_shape())
        if hasattr(layer, 'initialize'):
            layer.initialize(self.optimizer)
        self.layers.append(layer)
        return self

    def get_mini_batches(self, X, y=None, shuffle=True):
        n_samples = X.shape[0]
        indices = np.arange(n_samples)
        assert self.batch_size <= n_samples, "Batch size cannot be greater than the number of samples"
        if shuffle:
            np.random.shuffle(indices)
        for start in range(0, n_samples, self.batch_size):
            if y is not None:
                yield X[indices[start:start + self.batch_size]], y[indices[start:start + self.batch_size]]
            else:
                yield X[indices[start:start + self.batch_size]], None

    def _snapshot_params(self):
        snap = []
        for layer in self.layers:
            if hasattr(layer, 'weights') and hasattr(layer, 'biases'):
                snap.append({'weights': layer.weights.copy(), 'biases': layer.biases.copy()})
            else:
                snap.append(None)
        return snap

    def _restore_params(self, snap):
        for layer, s in zip(self.layers, snap):
            if s is None:
                continue
            layer.weights = s['weights']
            layer.biases = s['biases']

    def forward_propagation(self, X, training):
        output = X
        for layer in self.layers:
            output = layer.forward_propagation(output, training)
        return output

    def backward_propagation(self, output_error):
        error = output_error
        for layer in reversed(self.layers):
            error = layer.backward_propagation(error)
        return error

    def fit(self, dataset, val_dataset=None, patience: int = 0, min_delta: float = 0.0):
        X = dataset.X
        y = dataset.y
        if np.ndim(y) == 1 and self.loss.__class__.__name__ != 'SoftmaxCrossEntropy':
            y = np.expand_dims(y, axis=1)

        X_val, y_val = None, None
        if val_dataset is not None:
            X_val = val_dataset.X
            y_val = val_dataset.y
            if np.ndim(y_val) == 1 and self.loss.__class__.__name__ != 'SoftmaxCrossEntropy':
                y_val = np.expand_dims(y_val, axis=1)

        best_params = None
        best_val_loss = np.inf
        bad_epochs = 0
        self.history = {}

        for epoch in range(1, self.epochs + 1):
            output_x_, y_ = [], []
            for X_batch, y_batch in self.get_mini_batches(X, y):
                output = self.forward_propagation(X_batch, training=True)
                error = self.loss.derivative(y_batch, output)
                self.backward_propagation(error)
                output_x_.append(output)
                y_.append(y_batch)

            output_x_all = np.concatenate(output_x_)
            y_all = np.concatenate(y_)
            loss = self.loss.loss(y_all, output_x_all)

            if self.metric is not None:
                metric = self.metric(y_all, output_x_all)
                metric_s = f"{self.metric.__name__}: {metric:.4f}"
            else:
                metric_s = "NA"
                metric = 'NA'

            self.history[epoch] = {'loss': loss, 'metric': metric}

            if X_val is not None and y_val is not None:
                yv_pred = self.forward_propagation(X_val, training=False)
                val_loss = self.loss.loss(y_val, yv_pred)
                self.history[epoch]['val_loss'] = val_loss
                if self.metric is not None:
                    self.history[epoch]['val_metric'] = self.metric(y_val, yv_pred)

                if val_loss < (best_val_loss - min_delta):
                    best_val_loss = val_loss
                    best_params = self._snapshot_params()
                    bad_epochs = 0
                else:
                    bad_epochs += 1

                if patience and bad_epochs >= patience:
                    if self.verbose:
                        print(f"Early stopping at epoch {epoch}/{self.epochs}")
                    break

            if self.verbose:
                if 'val_loss' in self.history[epoch]:
                    print(f"Epoch {epoch}/{self.epochs} - loss: {loss:.4f} - {metric_s} - val_loss: {self.history[epoch]['val_loss']:.4f}")
                else:
                    print(f"Epoch {epoch}/{self.epochs} - loss: {loss:.4f} - {metric_s}")

        if best_params is not None:
            self._restore_params(best_params)
        return self

    def predict(self, dataset):
        return self.forward_propagation(dataset.X, training=False)

    def score(self, dataset, predictions):
        if self.metric is not None:
            return self.metric(dataset.y, predictions)
        else:
            raise ValueError("No metric specified for the neural network.")

## 9. Regressão Logística (Softmax) (`logistic_regression.py`)

In [9]:
def _softmax(logits):
    z = logits - np.max(logits, axis=1, keepdims=True)
    exp = np.exp(z)
    return exp / np.sum(exp, axis=1, keepdims=True)


def _one_hot(y, n_classes):
    y = np.asarray(y).astype(int).ravel()
    oh = np.zeros((len(y), n_classes), dtype=np.float32)
    oh[np.arange(len(y)), y] = 1.0
    return oh


class SoftmaxRegression:
    """Regressão logística multiclasse (softmax) treinada com minibatch GD."""

    def __init__(self, learning_rate=0.1, momentum=0.9, epochs=200,
                 batch_size=256, l2=0.0, verbose=False, seed=42):
        self.learning_rate = learning_rate
        self.momentum = momentum
        self.epochs = int(epochs)
        self.batch_size = int(batch_size)
        self.l2 = float(l2)
        self.verbose = verbose
        self.seed = seed
        self.W = None
        self.b = None
        self.w_opt = Optimizer(learning_rate=learning_rate, momentum=momentum)
        self.b_opt = Optimizer(learning_rate=learning_rate, momentum=momentum)
        self.history = {}

    def _iter_batches(self, X, y):
        n = X.shape[0]
        idx = np.arange(n)
        rng = np.random.default_rng(self.seed)
        rng.shuffle(idx)
        for start in range(0, n, self.batch_size):
            sel = idx[start:start + self.batch_size]
            yield X[sel], y[sel]

    def fit(self, X, y, n_classes=None, X_val=None, y_val=None, patience=0, min_delta=0.0):
        X = np.asarray(X)
        y = np.asarray(y).ravel().astype(int)
        n_samples, n_features = X.shape
        if n_classes is None:
            n_classes = int(np.max(y)) + 1

        rng = np.random.default_rng(self.seed)
        self.W = (rng.standard_normal((n_features, n_classes)).astype(np.float32)) * 0.01
        self.b = np.zeros((1, n_classes), dtype=np.float32)

        best = None
        best_val = np.inf
        bad_epochs = 0

        for epoch in range(1, self.epochs + 1):
            losses = []
            for Xb, yb in self._iter_batches(X, y):
                y_oh = _one_hot(yb, n_classes)
                logits = Xb @ self.W + self.b
                probs = _softmax(logits)
                p = np.clip(probs, 1e-15, 1 - 1e-15)
                loss = -np.mean(np.sum(y_oh * np.log(p), axis=1))
                if self.l2 > 0:
                    loss += 0.5 * self.l2 * np.sum(self.W * self.W)
                losses.append(loss)

                grad_logits = (probs - y_oh) / Xb.shape[0]
                grad_W = Xb.T @ grad_logits
                grad_b = np.sum(grad_logits, axis=0, keepdims=True)
                if self.l2 > 0:
                    grad_W += self.l2 * self.W
                self.W = self.w_opt.update(self.W, grad_W)
                self.b = self.b_opt.update(self.b, grad_b)

            train_loss = float(np.mean(losses)) if losses else float('nan')
            self.history[epoch] = {'loss': train_loss}

            val_loss = None
            if X_val is not None and y_val is not None:
                val_loss = self._loss(X_val, y_val, n_classes)
                self.history[epoch]['val_loss'] = val_loss
                if val_loss < (best_val - min_delta):
                    best_val = val_loss
                    best = (self.W.copy(), self.b.copy())
                    bad_epochs = 0
                else:
                    bad_epochs += 1
                if patience and bad_epochs >= patience:
                    if self.verbose:
                        print(f"Early stopping at epoch {epoch}")
                    break

            if self.verbose and (epoch == 1 or epoch % 10 == 0):
                msg = f"Epoch {epoch}/{self.epochs} - loss: {train_loss:.4f}"
                if val_loss is not None:
                    msg += f" - val_loss: {val_loss:.4f}"
                print(msg)

        if best is not None:
            self.W, self.b = best
        return self

    def _loss(self, X, y, n_classes):
        y = np.asarray(y).ravel().astype(int)
        y_oh = _one_hot(y, n_classes)
        probs = self.predict_proba(X)
        p = np.clip(probs, 1e-15, 1 - 1e-15)
        loss = -np.mean(np.sum(y_oh * np.log(p), axis=1))
        if self.l2 > 0:
            loss += 0.5 * self.l2 * np.sum(self.W * self.W)
        return float(loss)

    def predict_proba(self, X):
        return _softmax(np.asarray(X) @ self.W + self.b)

    def predict(self, X):
        return np.argmax(self.predict_proba(X), axis=1)

## 10. Vectorizadores de Texto (`text_vectorizer.py`)
CountVectorizer e TF-IDF implementados em NumPy.

In [10]:
_TOKEN_RE = re.compile(r"\b\w+\b", flags=re.UNICODE)


def simple_tokenize(text: str, lowercase: bool = True):
    if text is None:
        return []
    text = str(text)
    if lowercase:
        text = text.lower()
    return _TOKEN_RE.findall(text)


class CountVectorizerNumpy:
    """CountVectorizer mínimo implementado com numpy."""

    def __init__(self, max_features=5000, min_df=1, lowercase=True, dtype=np.float32):
        self.max_features = int(max_features) if max_features is not None else None
        self.min_df = int(min_df)
        self.lowercase = lowercase
        self.dtype = dtype
        self.vocab_ = None

    def fit(self, texts):
        doc_freq = {}
        for text in texts:
            tokens = set(simple_tokenize(text, lowercase=self.lowercase))
            for tok in tokens:
                doc_freq[tok] = doc_freq.get(tok, 0) + 1
        items = [(tok, df) for tok, df in doc_freq.items() if df >= self.min_df]
        items.sort(key=lambda x: (-x[1], x[0]))
        if self.max_features is not None:
            items = items[:self.max_features]
        self.vocab_ = {tok: i for i, (tok, _) in enumerate(items)}
        return self

    def transform(self, texts):
        if self.vocab_ is None:
            raise ValueError("Vectorizer not fitted. Call fit() first.")
        X = np.zeros((len(texts), len(self.vocab_)), dtype=self.dtype)
        for i, text in enumerate(texts):
            for tok in simple_tokenize(text, lowercase=self.lowercase):
                j = self.vocab_.get(tok)
                if j is not None:
                    X[i, j] += 1.0
        return X

    def fit_transform(self, texts):
        return self.fit(texts).transform(texts)


class TfidfVectorizerNumpy(CountVectorizerNumpy):
    """TF-IDF Vectorizer usando base CountVectorizer, implementado com numpy."""

    def __init__(self, max_features=5000, min_df=1, lowercase=True,
                 dtype=np.float32, norm="l2", smooth_idf=True):
        super().__init__(max_features=max_features, min_df=min_df, lowercase=lowercase, dtype=dtype)
        self.norm = norm
        self.smooth_idf = smooth_idf
        self.idf_ = None

    def fit(self, texts):
        super().fit(texts)
        n_docs = len(texts)
        df = np.zeros(len(self.vocab_), dtype=np.int32)
        for text in texts:
            seen = set()
            for tok in simple_tokenize(text, lowercase=self.lowercase):
                j = self.vocab_.get(tok)
                if j is None or j in seen:
                    continue
                df[j] += 1
                seen.add(j)
        if self.smooth_idf:
            self.idf_ = np.log((1.0 + n_docs) / (1.0 + df)) + 1.0
        else:
            self.idf_ = np.log(np.maximum(df, 1) / n_docs) + 1.0  # fix: was inverted
        self.idf_ = self.idf_.astype(self.dtype)
        return self

    def transform(self, texts):
        X = super().transform(texts)
        if self.idf_ is None:
            raise ValueError("Vectorizer not fitted. Call fit() first.")
        doc_len = np.maximum(np.sum(X, axis=1, keepdims=True), 1.0)
        tf = X / doc_len
        X_tfidf = tf * self.idf_[None, :]
        if self.norm == "l2":
            norms = np.maximum(np.linalg.norm(X_tfidf, axis=1, keepdims=True), 1e-12)
            X_tfidf = X_tfidf / norms
        elif self.norm is not None:
            raise ValueError("norm must be 'l2' or None")
        return X_tfidf.astype(self.dtype)

## 11. Utilitários de Treino de Modelos de Texto (`train_text_models.py`)

In [11]:
def load_split(csv_path: str):
    df = pd.read_csv(csv_path)
    if 'content' not in df.columns or 'model' not in df.columns:
        raise ValueError("CSV must have columns: content, model")
    return df['content'].astype(str).tolist(), df['model'].astype(str).tolist()


def load_examples(csv_path: str):
    """Carrega exemplos de dataset-exemplos.csv (ID;Text;Label)."""
    df = pd.read_csv(csv_path, sep=';')
    if 'Text' not in df.columns or 'Label' not in df.columns:
        raise ValueError("CSV must have columns: Text; Label")
    return df['Text'].astype(str).tolist(), df['Label'].astype(str).tolist()


def load_unlabeled(csv_path: str):
    """Carrega textos sem label de subm1.csv (ID;Text)."""
    df = pd.read_csv(csv_path, sep=';')
    if 'Text' not in df.columns:
        raise ValueError("CSV must have a Text column")
    ids = df['ID'].astype(str).tolist() if 'ID' in df.columns else list(range(len(df)))
    return ids, df['Text'].astype(str).tolist()


def encode_labels(train_labels, labels):
    classes = sorted(set(train_labels))
    class_to_id = {c: i for i, c in enumerate(classes)}
    y, keep = [], []
    for i, lab in enumerate(labels):
        if lab in class_to_id:
            y.append(class_to_id[lab])
            keep.append(i)
    return np.array(y, dtype=np.int64), keep, classes


def select_by_indices(items, indices):
    return [items[i] for i in indices]


def evaluate_predictions(y_true, y_pred, classes):
    y_true = np.asarray(y_true).ravel().astype(int)
    y_pred = np.asarray(y_pred).ravel().astype(int)
    acc = float(np.mean(y_true == y_pred))
    per_class = {}
    for k, name in enumerate(classes):
        mask = y_true == k
        if np.any(mask):
            per_class[name] = float(np.mean(y_pred[mask] == y_true[mask]))
    return acc, per_class

## 12. Pipeline de Treino
Ajusta os caminhos dos ficheiros CSV e corre o baseline (Softmax Regression) e/ou a DNN.

In [12]:
# ── Configuração ──────────────────────────────────────────────
TRAIN_CSV  = '../splits/train.csv'
VAL_CSV    = '../splits/val.csv'
TEST_CSV   = '../splits/test.csv'

VECTORIZER   = 'tfidf'   # 'count' ou 'tfidf'
MAX_FEATURES = 8000
MIN_DF       = 2

RUN = 'both'  # 'baseline', 'dnn' ou 'both'

# Baseline
LR_BASE     = 0.2
EPOCHS_BASE = 150
BATCH_BASE  = 512
L2_BASE     = 1e-4

# DNN
HIDDEN      = 256
DROPOUT     = 0.3
LR_DNN      = 0.05
EPOCHS_DNN  = 60
BATCH_DNN   = 256
L2_DNN      = 1e-4
PATIENCE    = 8

In [13]:
# ── Carregamento e vectorização ───────────────────────────────
train_texts, train_labels = load_split(TRAIN_CSV)
val_texts,   val_labels   = load_split(VAL_CSV)
test_texts,  test_labels  = load_split(TEST_CSV)

y_train, keep_train, classes = encode_labels(train_labels, train_labels)
y_val,   keep_val,   _       = encode_labels(train_labels, val_labels)
y_test,  keep_test,  _       = encode_labels(train_labels, test_labels)

train_texts = select_by_indices(train_texts, keep_train)
val_texts   = select_by_indices(val_texts,   keep_val)
test_texts  = select_by_indices(test_texts,  keep_test)

n_classes = len(classes)
print(f"Classes ({n_classes}): {classes}")

vec = TfidfVectorizerNumpy(max_features=MAX_FEATURES, min_df=MIN_DF) \
      if VECTORIZER == 'tfidf' \
      else CountVectorizerNumpy(max_features=MAX_FEATURES, min_df=MIN_DF)

X_train = vec.fit_transform(train_texts)
X_val   = vec.transform(val_texts)
X_test  = vec.transform(test_texts)

print(f"X_train: {X_train.shape}  X_val: {X_val.shape}  X_test: {X_test.shape}")

Classes (5): ['Anthropic', 'Google', 'Human', 'Meta', 'OpenAI']
X_train: (41809, 8000)  X_val: (8958, 8000)  X_test: (8962, 8000)


In [14]:
# ── Baseline: Softmax Regression ─────────────────────────────
if RUN in ('baseline', 'both'):
    print("\n== Baseline: Softmax Regression ==")
    base = SoftmaxRegression(
        learning_rate=LR_BASE, epochs=EPOCHS_BASE,
        batch_size=BATCH_BASE, l2=L2_BASE, verbose=True,
    )
    base.fit(X_train, y_train, n_classes=n_classes,
             X_val=X_val, y_val=y_val, patience=10, min_delta=1e-4)

    pred_val  = base.predict(X_val)
    pred_test = base.predict(X_test)
    acc_val,  _         = evaluate_predictions(y_val,  pred_val,  classes)
    acc_test, per_class = evaluate_predictions(y_test, pred_test, classes)

    print(f"Val accuracy:  {acc_val:.4f}")
    print(f"Test accuracy: {acc_test:.4f}")
    print("Per-class test accuracy:")
    for k, v in per_class.items():
        print(f"  {k}: {v:.4f}")

    # Avaliação no dataset-exemplos.csv
    examples_path = '../dataset-exemplos.csv'
    if os.path.exists(examples_path):
        ex_texts, ex_labels = load_examples(examples_path)
        y_ex, keep_ex, _ = encode_labels(train_labels, ex_labels)
        ex_texts = select_by_indices(ex_texts, keep_ex)
        if len(ex_texts) > 0:
            X_ex = vec.transform(ex_texts)
            pred_ex = base.predict(X_ex)
            acc_ex, per_class_ex = evaluate_predictions(y_ex, pred_ex, classes)
            print("\n== Baseline on dataset-exemplos.csv ==")
            print(f"Accuracy: {acc_ex:.4f}")
            print("Per-class accuracy:")
            for k, v in per_class_ex.items():
                print(f"  {k}: {v:.4f}")

    # Previsões para subm1.csv
    subm_path = '../subm1.csv'
    if os.path.exists(subm_path):
        ids_sub, texts_sub = load_unlabeled(subm_path)
        if len(texts_sub) > 0:
            X_sub = vec.transform(texts_sub)
            pred_sub_idx = base.predict(X_sub)
            pred_sub_labels = [classes[i] for i in pred_sub_idx]
            out_df = pd.DataFrame({
                'ID': ids_sub,
                'Text': texts_sub,
                'PredictedModel': pred_sub_labels,
            })
            out_path = '../subm1_predictions_baseline.csv'
            out_df.to_csv(out_path, index=False)
            print(f"Baseline predictions written to {out_path}")


== Baseline: Softmax Regression ==
Epoch 1/150 - loss: 1.5586 - val_loss: 1.5360
Epoch 10/150 - loss: 1.3463 - val_loss: 1.3376
Epoch 20/150 - loss: 1.1879 - val_loss: 1.1832
Epoch 30/150 - loss: 1.0762 - val_loss: 1.0745
Epoch 40/150 - loss: 0.9941 - val_loss: 0.9948
Epoch 50/150 - loss: 0.9315 - val_loss: 0.9342
Epoch 60/150 - loss: 0.8825 - val_loss: 0.8867
Epoch 70/150 - loss: 0.8431 - val_loss: 0.8487
Epoch 80/150 - loss: 0.8110 - val_loss: 0.8177
Epoch 90/150 - loss: 0.7843 - val_loss: 0.7920
Epoch 100/150 - loss: 0.7619 - val_loss: 0.7706
Epoch 110/150 - loss: 0.7430 - val_loss: 0.7525
Epoch 120/150 - loss: 0.7268 - val_loss: 0.7370
Epoch 130/150 - loss: 0.7128 - val_loss: 0.7238
Epoch 140/150 - loss: 0.7007 - val_loss: 0.7123
Epoch 150/150 - loss: 0.6902 - val_loss: 0.7024
Val accuracy:  0.8771
Test accuracy: 0.8838
Per-class test accuracy:
  Anthropic: 0.8538
  Google: 0.8493
  Human: 0.9390
  Meta: 0.8621
  OpenAI: 0.8592

== Baseline on dataset-exemplos.csv ==
Accuracy: 0.4

In [15]:
# ── DNN (numpy) ───────────────────────────────────────────────
if RUN in ('dnn', 'both'):
    print("\n== DNN (numpy) ==")
    ds_train = Data(X_train.astype(np.float32), y_train)
    ds_val   = Data(X_val.astype(np.float32),   y_val)
    ds_test  = Data(X_test.astype(np.float32),  y_test)

    net = NeuralNetwork(
        epochs=EPOCHS_DNN, batch_size=BATCH_DNN,
        learning_rate=LR_DNN, verbose=True,
        loss=SoftmaxCrossEntropy, metric=softmax_accuracy,
    )
    n_features = ds_train.X.shape[1]
    net.add(DenseLayer(HIDDEN, (n_features,), l2_lambda=L2_DNN))
    net.add(ReLUActivation())
    net.add(DropoutLayer(p_drop=DROPOUT))
    net.add(DenseLayer(n_classes, l2_lambda=L2_DNN))

    net.fit(ds_train, val_dataset=ds_val, patience=PATIENCE, min_delta=1e-4)

    logits_test = net.predict(ds_test)
    pred_test   = np.argmax(logits_test, axis=1)
    acc_test, per_class = evaluate_predictions(y_test, pred_test, classes)

    print(f"Test accuracy: {acc_test:.4f}")
    print("Per-class test accuracy:")
    for k, v in per_class.items():
        print(f"  {k}: {v:.4f}")

    # Avaliação da DNN no dataset-exemplos.csv
    examples_path = '../dataset-exemplos.csv'
    if os.path.exists(examples_path):
        ex_texts, ex_labels = load_examples(examples_path)
        y_ex, keep_ex, _ = encode_labels(train_labels, ex_labels)
        ex_texts = select_by_indices(ex_texts, keep_ex)
        if len(ex_texts) > 0:
            X_ex = vec.transform(ex_texts).astype(np.float32)
            ds_ex = Data(X_ex, y_ex)
            logits_ex = net.predict(ds_ex)
            pred_ex = np.argmax(logits_ex, axis=1)
            acc_ex, per_class_ex = evaluate_predictions(y_ex, pred_ex, classes)
            print("\n== DNN on dataset-exemplos.csv ==")
            print(f"Accuracy: {acc_ex:.4f}")
            print("Per-class accuracy:")
            for k, v in per_class_ex.items():
                print(f"  {k}: {v:.4f}")

    # Previsões da DNN para subm1.csv
    subm_path = '../subm1.csv'
    if os.path.exists(subm_path):
        ids_sub, texts_sub = load_unlabeled(subm_path)
        if len(texts_sub) > 0:
            X_sub = vec.transform(texts_sub).astype(np.float32)
            ds_sub = Data(X_sub)
            logits_sub = net.predict(ds_sub)
            pred_sub_idx = np.argmax(logits_sub, axis=1)
            pred_sub_labels = [classes[i] for i in pred_sub_idx]
            out_df = pd.DataFrame({
                'ID': ids_sub,
                'Text': texts_sub,
                'PredictedModel': pred_sub_labels,
            })
            out_path = '../subm1_predictions_dnn.csv'
            out_df.to_csv(out_path, index=False)
            print(f"DNN predictions written to {out_path}")


== DNN (numpy) ==
Epoch 1/60 - loss: 1.7660 - softmax_accuracy: 0.2709 - val_loss: 1.5763
Epoch 2/60 - loss: 1.5950 - softmax_accuracy: 0.3270 - val_loss: 1.4661
Epoch 3/60 - loss: 1.4982 - softmax_accuracy: 0.3649 - val_loss: 1.3880
Epoch 4/60 - loss: 1.4229 - softmax_accuracy: 0.4025 - val_loss: 1.3237
Epoch 5/60 - loss: 1.3621 - softmax_accuracy: 0.4306 - val_loss: 1.2667
Epoch 6/60 - loss: 1.3131 - softmax_accuracy: 0.4559 - val_loss: 1.2151
Epoch 7/60 - loss: 1.2664 - softmax_accuracy: 0.4770 - val_loss: 1.1661
Epoch 8/60 - loss: 1.2224 - softmax_accuracy: 0.4970 - val_loss: 1.1213
Epoch 9/60 - loss: 1.1864 - softmax_accuracy: 0.5142 - val_loss: 1.0806
Epoch 10/60 - loss: 1.1525 - softmax_accuracy: 0.5293 - val_loss: 1.0427
Epoch 11/60 - loss: 1.1187 - softmax_accuracy: 0.5440 - val_loss: 1.0075
Epoch 12/60 - loss: 1.0886 - softmax_accuracy: 0.5561 - val_loss: 0.9755
Epoch 13/60 - loss: 1.0575 - softmax_accuracy: 0.5708 - val_loss: 0.9459
Epoch 14/60 - loss: 1.0322 - softmax_accu